# 第5章：特征工程 - Alpha因子 ⭐

## 本章学习目标

- 深入理解 Alpha158 特征集设计
- 掌握表达式引擎使用
- 能够开发自定义因子
- 学会因子相关性分析

---

## 5.1 Alpha158 特征集概述

Alpha158 是 qlib 内置的高质量因子集，包含 158 个特征，广泛应用于股票收益预测模型。

### 因子分类

| 类别 | 数量 | 说明 |
|------|------|------|
| KBAR | 30+ | K 线形态因子 |
| KDJ | 18+ | 随机指标变体 |
| RSV | 18+ | 相对强弱因子 |
| MA | 30+ | 均线类因子 |
| MACD | 18+ | 趋势类因子 |
| RSI | 18+ | 相对强弱指数 |
| PSY | 18+ | 心理线因子 |
| BIAS | 18+ | 乖离率因子 |

### 设计理念

1. **多时间窗口**：覆盖 5、10、20、30、60 日等多个周期
2. **多维度**：价格、成交量、波动率、动量等
3. **正交化**：因子之间相关性较低
4. **可解释**：每个因子有明确的金融含义

In [ ]:
import qlib
from qlib.data import D
from qlib.contrib.data.handler import Alpha158, Alpha360
from qlib.data.ops import Operators
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 初始化 qlib
qlib.init(
    provider_uri="~/.qlib/qlib_data/cn_data",
    region="cn",
)

print("Qlib 初始化成功")

## 5.2 Alpha158 因子详解

### 5.2.1 创建 Alpha158 Handler

In [ ]:
# 创建 Alpha158 handler
# 注意：cn_data 数据最晚到 2020-09-25
handler = Alpha158(
    instruments="csi300",
    start_time="2016-01-01",
    end_time="2020-09-25",  # 数据最晚日期
    freq="day",
)

# 获取特征数据
df = handler.fetch()

print(f"特征数据形状: {df.shape}")
print(f"特征数量: {len(df.columns)}")

In [ ]:
# 查看所有特征名称
print("Alpha158 特征列表:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:3d}. {col}")

In [ ]:
# 按类别分组特征
def categorize_features(features):
    """将特征按类别分组"""
    categories = {
        'KBAR': [],
        'KDJ': [],
        'RSV': [],
        'MA': [],
        'MACD': [],
        'RSI': [],
        'PSY': [],
        'BIAS': [],
        '其他': [],
    }
    
    for f in features:
        categorized = False
        for cat in categories:
            if cat != '其他' and cat in f.upper():
                categories[cat].append(f)
                categorized = True
                break
        if not categorized:
            categories['其他'].append(f)
    
    return categories

categories = categorize_features(df.columns.tolist())

print("Alpha158 因子分类统计:")
print("-" * 40)
for cat, features in categories.items():
    print(f"{cat:10s}: {len(features):3d} 个")

### 5.2.2 查看特征数据

In [ ]:
# 查看前几行数据
df.head()

In [ ]:
# 数据基本统计
df.describe()

In [ ]:
# 查看缺失值情况
missing_stats = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_pct': df.isna().sum() / len(df) * 100,
}).sort_values('missing_count', ascending=False)

print("缺失值最多的 10 个特征:")
missing_stats.head(10)

## 5.3 表达式引擎

Qlib 的表达式引擎允许用户使用字符串表达式定义因子，系统会自动解析和计算。

### 5.3.1 常用操作符

In [ ]:
# 查看可用的操作符
from qlib.data import ops
import inspect

print("Qlib 表达式操作符:")
print("=" * 60)

# 获取 Operators 类的所有方法
operators = []
for name in dir(ops):
    obj = getattr(ops, name)
    if inspect.isclass(obj) and name[0].isupper():
        operators.append(name)

# 分类显示
print("\n引用类操作符 (Ref, Shift):")
ref_ops = [op for op in operators if 'Ref' in op or 'Shift' in op]
for op in sorted(ref_ops):
    print(f"  - {op}")

print("\n统计类操作符:")
stat_ops = [op for op in operators if any(s in op for s in ['Mean', 'Std', 'Sum', 'Max', 'Min', 'Var', 'Med'])]
for op in sorted(stat_ops):
    print(f"  - {op}")

print("\n相关类操作符:")
corr_ops = [op for op in operators if 'Corr' in op or 'Cov' in op]
for op in sorted(corr_ops):
    print(f"  - {op}")

print("\n其他操作符:")
other_ops = [op for op in operators if op not in ref_ops + stat_ops + corr_ops]
for op in sorted(other_ops)[:20]:
    print(f"  - {op}")

### 5.3.2 表达式示例

In [ ]:
# 常用因子表达式示例
factor_expressions = {
    # 价格类
    "收盘价": "$close",
    "开盘价": "$open",
    "最高价": "$high",
    "最低价": "$low",
    "成交量": "$volume",
    
    # 移动平均
    "MA5": "Mean($close, 5)",
    "MA10": "Mean($close, 10)",
    "MA20": "Mean($close, 20)",
    "MA60": "Mean($close, 60)",
    
    # 动量
    "动量_5日": "$close / Ref($close, 5) - 1",
    "动量_10日": "$close / Ref($close, 10) - 1",
    "动量_20日": "$close / Ref($close, 20) - 1",
    
    # 波动率
    "波动率_5日": "Std($close / Ref($close, 1) - 1, 5)",
    "波动率_20日": "Std($close / Ref($close, 1) - 1, 20)",
    
    # 振幅
    "振幅_5日": "Max($high, 5) / Min($low, 5) - 1",
    
    # 成交量变化
    "量比_5日": "$volume / Mean($volume, 5)",
    
    # 相对指标
    "RSV_5日": "($close - Min($low, 5)) / (Max($high, 5) - Min($low, 5))",
}

print("因子表达式示例:")
print("=" * 60)
for name, expr in factor_expressions.items():
    print(f"{name:15s}: {expr}")

In [ ]:
# 使用表达式获取因子数据
df_factors = D.features(
    instruments=["SH600000"],  # 必须是列表
    fields=[
        "$close",
        "Mean($close, 5)",
        "Mean($close, 20)",
        "$close / Ref($close, 5) - 1",
        "Std($close / Ref($close, 1) - 1, 20)",
    ],
    start_time="2020-01-01",
    end_time="2020-09-25",
)

# 重命名列
df_factors.columns = ['close', 'ma5', 'ma20', 'momentum_5d', 'volatility_20d']

print("因子数据:")
df_factors.head(20)

### 5.3.3 复杂表达式

In [ ]:
# 复杂因子表达式
complex_factors = D.features(
    instruments=["SH600000"],  # 必须是列表
    fields=[
        # 布林带宽度
        "(Mean($close, 20) + 2 * Std($close, 20) - (Mean($close, 20) - 2 * Std($close, 20))) / Mean($close, 20)",
        
        # 价格相对位置 (Price Relative Position)
        "($close - Min($low, 20)) / (Max($high, 20) - Min($low, 20))",
        
        # 成交量加权价格动量
        "($vwap - Ref($vwap, 5)) / Ref($vwap, 5)",
        
        # 价格与均线偏离度
        "($close - Mean($close, 20)) / Mean($close, 20)",
    ],
    start_time="2020-01-01",
    end_time="2020-09-25",
)

complex_factors.columns = ['boll_width', 'price_rel_pos', 'vwap_momentum', 'ma_deviation']

print("复杂因子数据:")
complex_factors.head(20)

## 5.4 自定义因子开发

### 5.4.1 通过表达式定义因子

In [ ]:
# 定义自定义因子表达式
custom_factor_expressions = {
    # 改进的动量因子：考虑波动率
    "risk_adj_momentum": "($close / Ref($close, 20) - 1) / Std($close / Ref($close, 1) - 1, 20)",
    
    # 成交量加权收益率
    "vol_weighted_return": "Sum($close / Ref($close, 1) - 1, 5) * Mean($volume, 5) / Mean($volume, 20)",
    
    # 价格效率 (Price Efficiency)
    "price_efficiency": "Abs($close - Ref($close, 10)) / (Sum(Abs($close - Ref($close, 1)), 10) + 0.0001)",
    
    # 相对成交量
    "relative_volume": "$volume / Mean($volume, 20)",
    
    # 高低点比率
    "high_low_ratio": "Mean($high, 5) / Mean($low, 5)",
}

# 获取自定义因子数据
df_custom = D.features(
    instruments=["SH600000"],  # 必须是列表
    fields=list(custom_factor_expressions.values()),
    start_time="2020-01-01",
    end_time="2020-09-25",
)

df_custom.columns = list(custom_factor_expressions.keys())

print("自定义因子数据:")
df_custom.head(20)

### 5.4.2 构建自定义 DataHandler

In [ ]:
from qlib.data.dataset.handler import DataHandlerLP

class CustomFactorHandler(DataHandlerLP):
    """自定义因子 DataHandler"""
    
    def __init__(
        self,
        instruments="csi300",
        start_time=None,
        end_time=None,
        freq="day",
        infer_processors=None,
        **kwargs
    ):
        # 定义因子表达式
        self.fields = [
            # 基础价格因子
            "$close",
            "$volume",
            "$vwap",
            
            # 动量因子
            "$close / Ref($close, 5) - 1",
            "$close / Ref($close, 10) - 1",
            "$close / Ref($close, 20) - 1",
            
            # 波动率因子
            "Std($close / Ref($close, 1) - 1, 5)",
            "Std($close / Ref($close, 1) - 1, 10)",
            "Std($close / Ref($close, 1) - 1, 20)",
            
            # 成交量因子
            "$volume / Mean($volume, 5)",
            "$volume / Mean($volume, 20)",
            
            # 技术指标
            "($close - Min($low, 14)) / (Max($high, 14) - Min($low, 14))",  # KDV 类似
            "($close - Mean($close, 20)) / Mean($close, 20)",  # BIAS
        ]
        
        # 定义列名
        self.col_names = [
            'close', 'volume', 'vwap',
            'mom_5d', 'mom_10d', 'mom_20d',
            'vol_5d', 'vol_10d', 'vol_20d',
            'vol_ratio_5d', 'vol_ratio_20d',
            'rsv_14d', 'bias_20d',
        ]
        
        super().__init__(
            instruments=instruments,
            start_time=start_time,
            end_time=end_time,
            freq=freq,
            infer_processors=infer_processors,
            **kwargs
        )

# 创建自定义 handler
# 注意：cn_data 数据最晚到 2020-09-25
custom_handler = CustomFactorHandler(
    instruments="csi300",
    start_time="2016-01-01",
    end_time="2020-09-25",  # 数据最晚日期
)

print(f"自定义 Handler 创建成功")
print(f"因子数量: {len(custom_handler.fields)}")

In [ ]:
# 获取数据
df_custom_handler = custom_handler.fetch()

print(f"数据形状: {df_custom_handler.shape}")
df_custom_handler.head()

## 5.5 因子相关性分析

因子之间的相关性是因子选择的重要指标，高相关性的因子会带来信息冗余。

In [ ]:
# 计算因子相关性矩阵
# 选取部分因子进行分析
selected_features = df.columns[:30].tolist()
df_selected = df[selected_features].dropna()

# 计算相关系数
corr_matrix = df_selected.corr()

print(f"相关性矩阵形状: {corr_matrix.shape}")
corr_matrix.head()

In [ ]:
# 绘制相关性热力图
import seaborn as sns

plt.figure(figsize=(14, 12))

# 只显示部分因子
sns.heatmap(
    corr_matrix.iloc[:20, :20],
    cmap='RdBu_r',
    center=0,
    annot=False,
    square=True,
    linewidths=0.5,
)

plt.title('因子相关性热力图 (前20个因子)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 分析高相关性因子对
def find_high_corr_pairs(corr_matrix, threshold=0.7):
    """找出高相关性的因子对"""
    high_corr = []
    n = len(corr_matrix)
    
    for i in range(n):
        for j in range(i + 1, n):
            if abs(corr_matrix.iloc[i, j]) > threshold:
                high_corr.append({
                    'factor_1': corr_matrix.index[i],
                    'factor_2': corr_matrix.columns[j],
                    'correlation': corr_matrix.iloc[i, j],
                })
    
    return pd.DataFrame(high_corr).sort_values('correlation', key=abs, ascending=False)

high_corr_df = find_high_corr_pairs(corr_matrix, threshold=0.7)

print(f"高相关性因子对数量: {len(high_corr_df)}")
print("\n相关性最高的 10 对因子:")
high_corr_df.head(10)

In [ ]:
# 因子相关性分布
plt.figure(figsize=(10, 5))

# 提取上三角相关系数
upper_tri = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
).stack()

plt.hist(upper_tri, bins=50, edgecolor='black', alpha=0.7)
plt.axvline(x=0.7, color='red', linestyle='--', label='阈值 0.7')
plt.axvline(x=-0.7, color='red', linestyle='--')
plt.xlabel('相关系数')
plt.ylabel('频数')
plt.title('因子相关系数分布')
plt.legend()
plt.show()

print(f"\n相关系数统计:")
print(f"  均值: {upper_tri.mean():.4f}")
print(f"  标准差: {upper_tri.std():.4f}")
print(f"  绝对值 > 0.7 的比例: {(abs(upper_tri) > 0.7).mean():.2%}")

## 5.6 因子重要性分析

In [ ]:
# 使用 LightGBM 模型分析因子重要性
from qlib.contrib.model.gbdt import LGBModel
from qlib.data.dataset import DatasetH

# 创建数据集
# 注意：cn_data 数据最晚到 2020-09-25
dataset = DatasetH(
    handler={
        "class": "Alpha158",
        "module_path": "qlib.contrib.data.handler",
        "kwargs": {
            "start_time": "2016-01-01",
            "end_time": "2020-09-25",  # 数据最晚日期
            "fit_start_time": "2016-01-01",
            "fit_end_time": "2019-12-31",
            "instruments": "csi300",
        },
    },
    segments={
        "train": ("2016-01-01", "2019-12-31"),
        "test": ("2020-01-01", "2020-09-25"),  # 数据最晚日期
    },
)

# 创建模型
model = LGBModel(
    loss="mse",
    learning_rate=0.05,
    num_leaves=64,
    max_depth=6,
    n_estimators=100,
)

print("开始训练模型...")
model.fit(dataset)
print("模型训练完成")

In [ ]:
# 获取特征重要性
importance = model.get_feature_importance()

# 排序
importance_sorted = importance.sort_values(ascending=False)

print("特征重要性排名 (Top 20):")
print(importance_sorted.head(20))

In [ ]:
# 可视化特征重要性
plt.figure(figsize=(12, 8))

top_n = 30
importance_top = importance_sorted.head(top_n)

plt.barh(range(len(importance_top)), importance_top.values, color='steelblue')
plt.yticks(range(len(importance_top)), importance_top.index)
plt.xlabel('重要性')
plt.ylabel('特征')
plt.title(f'特征重要性 Top {top_n}')
plt.tight_layout()
plt.show()

## 5.7 实践练习

### 练习目标

1. 分析 Alpha158 因子组成结构
2. 使用表达式计算各类因子
3. 实现 5 个自定义技术因子
4. 因子相关性分析

In [ ]:
# 练习1: 实现 RSI 因子
# RSI = 100 - 100 / (1 + RS)
# RS = 平均上涨幅度 / 平均下跌幅度

# 你的代码：使用表达式定义 RSI 因子



# 参考答案（手动计算方式）
# df = D.features(instruments=["SH600000"], fields=["$close"], start_time="2020-01-01", end_time="2020-09-25")
# delta = df["$close"].diff()
# gain = delta.where(delta > 0, 0).rolling(14).mean()
# loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
# rs = gain / loss
# df["rsi_14"] = 100 - (100 / (1 + rs))

In [ ]:
# 练习2: 实现 MACD 因子
# MACD = EMA(12) - EMA(26)
# Signal = EMA(MACD, 9)

# 你的代码



# 提示：qlib 表达式引擎可能不支持 EMA，需要手动计算

In [ ]:
# 练习3: 创建一个包含 10 个自定义因子的 DataHandler
# 包括：动量因子、波动率因子、成交量因子、技术指标因子

# 你的代码



# class MyFactorHandler(DataHandlerLP):
#     def __init__(self, ...):
#         self.fields = [
#             # 添加你的因子表达式
#         ]
#         ...

In [ ]:
# 练习4: 计算自定义因子的相关性矩阵
# 并找出相关性 > 0.8 的因子对

# 你的代码



# 提示：
# corr_matrix = df.corr()
# 使用 find_high_corr_pairs 函数

## 5.8 本章小结

本章我们学习了：

1. **Alpha158 特征集**：
   - 158 个因子的分类与设计理念
   - 多时间窗口、多维度、正交化

2. **表达式引擎**：
   - 常用操作符：`Ref`, `Mean`, `Std`, `Sum`, `Max`, `Min`
   - 表达式语法与使用

3. **自定义因子开发**：
   - 通过表达式定义因子
   - 构建自定义 DataHandler

4. **因子分析**：
   - 相关性分析
   - 重要性分析

### 关键操作符速查

| 操作符 | 说明 | 示例 |
|--------|------|------|
| `Ref` | 历史引用 | `Ref($close, 5)` |
| `Mean` | 移动平均 | `Mean($close, 20)` |
| `Std` | 标准差 | `Std($close, 20)` |
| `Sum` | 求和 | `Sum($volume, 5)` |
| `Max` | 最大值 | `Max($high, 20)` |
| `Min` | 最小值 | `Min($low, 20)` |
| `Abs` | 绝对值 | `Abs($close - Ref($close, 1))` |
| `Rank` | 排名 | `Rank($close)` |
| `Corr` | 相关系数 | `Corr($close, $volume, 20)` |

### 下一章预告

下一章我们将学习高级数据处理，包括：
- PIT 数据的使用
- 数据重采样机制
- 滚动数据处理